In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install segmentation-models-pytorch -q

In [12]:
import segmentation_models_pytorch as smp

WORLDCOVER_CLASSES = {10:0,20:1,30:2,40:3,50:4,60:5,70:6,80:7,90:8,95:9,100:10}
NUM_CLASSES = len(WORLDCOVER_CLASSES)

def remap_labels(label_array):
    remapped = label_array.copy()
    for raw_val, class_idx in WORLDCOVER_CLASSES.items():
        remapped[label_array == raw_val] = class_idx
    return remapped

def build_rgb_baseline(num_classes=NUM_CLASSES, encoder="resnet34"):
    return smp.Unet(encoder_name=encoder, encoder_weights="imagenet", in_channels=3, classes=num_classes)

def build_multispectral_model(num_classes=NUM_CLASSES, encoder="resnet34", in_channels=13):
    return smp.Unet(encoder_name=encoder, encoder_weights="imagenet", in_channels=in_channels, classes=num_classes)

In [13]:
import os, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

PATCH_DIR = "/content/drive/MyDrive/patches"
CHECKPOINT_DIR = "/content/drive/MyDrive/patches/checkpoints"
BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = False  # avoids intermittent CUDNN_STATUS_INTERNAL_ERROR on Colab T4
print("Device:", DEVICE)

class TileDataset(Dataset):
    def __init__(self, patch_dir, split, channels="all"):
        self.image_dir = os.path.join(patch_dir, split, "images")
        self.label_dir = os.path.join(patch_dir, split, "labels")
        self.files = sorted(os.listdir(self.image_dir))
        self.channels = channels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        image = np.load(os.path.join(self.image_dir, fname))
        label = np.load(os.path.join(self.label_dir, fname))
        if self.channels == "rgb":
            image = image[[2, 1, 0], :, :]
        label = remap_labels(label)  # raw WorldCover codes (10-100) -> 0-10 class indices
        return torch.from_numpy(image).float(), torch.from_numpy(label).long()

Device: cuda


In [15]:
def compute_class_weights(dataset, num_classes):
    counts = np.zeros(num_classes)
    for i in range(len(dataset)):
        _, label = dataset[i]
        for c in label.unique().tolist():
            counts[c] += (label == c).sum().item()
    total = counts.sum()
    weights = np.zeros(num_classes)
    nonzero = counts > 0
    weights[nonzero] = total / (num_classes * counts[nonzero])
    weights = np.clip(weights, 0, 10)
    return torch.tensor(weights, dtype=torch.float32)

def train_model(model, model_name, in_channels_mode, class_weights=None):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    train_ds = TileDataset(PATCH_DIR, "train", channels=in_channels_mode)
    val_ds = TileDataset(PATCH_DIR, "val", channels=in_channels_mode)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE)) if class_weights is not None else nn.CrossEntropyLoss()
    best_val_loss = float("inf")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        print(f"[{model_name}] Epoch {epoch+1}/{EPOCHS} — train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}")

        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"{model_name}_latest.pt"))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"{model_name}_best.pt"))
            print(f"  New best saved (val_loss: {val_loss:.4f})")
    return model

In [22]:
rgb_model = build_rgb_baseline()
rgb_model = train_model(rgb_model, "rgb_baseline", in_channels_mode="rgb", class_weights=None)

[rgb_baseline] Epoch 1/30 — train_loss: 2.1474, val_loss: 1.7797
  New best saved (val_loss: 1.7797)
[rgb_baseline] Epoch 2/30 — train_loss: 1.4941, val_loss: 1.3797
  New best saved (val_loss: 1.3797)
[rgb_baseline] Epoch 3/30 — train_loss: 1.1806, val_loss: 1.1650
  New best saved (val_loss: 1.1650)
[rgb_baseline] Epoch 4/30 — train_loss: 1.0173, val_loss: 0.9555
  New best saved (val_loss: 0.9555)
[rgb_baseline] Epoch 5/30 — train_loss: 0.8970, val_loss: 0.9025
  New best saved (val_loss: 0.9025)
[rgb_baseline] Epoch 6/30 — train_loss: 0.8200, val_loss: 0.8640
  New best saved (val_loss: 0.8640)
[rgb_baseline] Epoch 7/30 — train_loss: 0.7737, val_loss: 0.7965
  New best saved (val_loss: 0.7965)
[rgb_baseline] Epoch 8/30 — train_loss: 0.7399, val_loss: 0.7869
  New best saved (val_loss: 0.7869)
[rgb_baseline] Epoch 9/30 — train_loss: 0.7170, val_loss: 0.7715
  New best saved (val_loss: 0.7715)
[rgb_baseline] Epoch 10/30 — train_loss: 0.6824, val_loss: 0.7526
  New best saved (val_los

In [23]:
ms_model = build_multispectral_model()
ms_model = train_model(ms_model, "multispectral", in_channels_mode="all", class_weights=None)

[multispectral] Epoch 1/30 — train_loss: 2.0179, val_loss: 1.6092
  New best saved (val_loss: 1.6092)
[multispectral] Epoch 2/30 — train_loss: 1.3791, val_loss: 1.2984
  New best saved (val_loss: 1.2984)
[multispectral] Epoch 3/30 — train_loss: 1.1252, val_loss: 1.1895
  New best saved (val_loss: 1.1895)
[multispectral] Epoch 4/30 — train_loss: 0.9717, val_loss: 1.0573
  New best saved (val_loss: 1.0573)
[multispectral] Epoch 5/30 — train_loss: 0.8733, val_loss: 0.8792
  New best saved (val_loss: 0.8792)
[multispectral] Epoch 6/30 — train_loss: 0.8185, val_loss: 0.8357
  New best saved (val_loss: 0.8357)
[multispectral] Epoch 7/30 — train_loss: 0.7687, val_loss: 0.7842
  New best saved (val_loss: 0.7842)
[multispectral] Epoch 8/30 — train_loss: 0.7474, val_loss: 0.8778
[multispectral] Epoch 9/30 — train_loss: 0.7221, val_loss: 0.9226
[multispectral] Epoch 10/30 — train_loss: 0.6818, val_loss: 0.7598
  New best saved (val_loss: 0.7598)
[multispectral] Epoch 11/30 — train_loss: 0.6519, v

EVALUATE...

In [24]:
import os, numpy as np, torch
from collections import Counter

CLASS_NAMES = ["Tree cover","Shrubland","Grassland","Cropland","Built-up",
               "Bare/sparse veg","Snow/ice","Water","Wetland","Mangroves","Moss/lichen"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
def check_class_distribution(dataset):
    counts = Counter()
    for i in range(len(dataset)):
        _, label = dataset[i]
        for c in label.unique().tolist():
            counts[c] += (label == c).sum().item()
    total = sum(counts.values())
    print("Class pixel distribution in test set:")
    for c in range(NUM_CLASSES):
        pct = 100 * counts.get(c, 0) / total if total else 0
        print(f"  {CLASS_NAMES[c]:20s}: {pct:5.2f}%")
    return counts

test_ds_ms = TileDataset(PATCH_DIR, "test", channels="all")
check_class_distribution(test_ds_ms)

Class pixel distribution in test set:
  Tree cover          : 37.60%
  Shrubland           :  0.01%
  Grassland           :  2.31%
  Cropland            : 19.89%
  Built-up            : 31.12%
  Bare/sparse veg     :  0.62%
  Snow/ice            :  0.00%
  Water               :  7.77%
  Wetland             :  0.68%
  Mangroves           :  0.00%
  Moss/lichen         :  0.00%


Counter({0: 714583,
         2: 43841,
         3: 378071,
         4: 591438,
         7: 147652,
         5: 11851,
         8: 12887,
         1: 221})

In [26]:
from torch.utils.data import DataLoader

def compute_iou_per_class(model, dataset, batch_size=8):
    loader = DataLoader(dataset, batch_size=batch_size)
    model.eval()
    intersection = np.zeros(NUM_CLASSES)
    union = np.zeros(NUM_CLASSES)
    correct = 0
    total = 0
    confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            labels = labels.numpy()
            for c in range(NUM_CLASSES):
                pred_c = preds == c
                label_c = labels == c
                intersection[c] += np.logical_and(pred_c, label_c).sum()
                union[c] += np.logical_or(pred_c, label_c).sum()
            correct += (preds == labels).sum()
            total += labels.size
            for t, p in zip(labels.flatten(), preds.flatten()):
                confusion[t, p] += 1

    iou_per_class = intersection / np.maximum(union, 1)
    pixel_accuracy = correct / total
    mean_iou = iou_per_class[union > 0].mean()
    return {"iou_per_class": iou_per_class, "mean_iou": mean_iou,
            "pixel_accuracy": pixel_accuracy, "confusion": confusion}

def print_results(name, results):
    print(f"\n=== {name} ===")
    print(f"Pixel accuracy: {results['pixel_accuracy']:.4f}")
    print(f"Mean IoU (present classes): {results['mean_iou']:.4f}")
    print("Per-class IoU:")
    for c in range(NUM_CLASSES):
        print(f"  {CLASS_NAMES[c]:20s}: {results['iou_per_class'][c]:.4f}")

In [27]:
test_ds_rgb = TileDataset(PATCH_DIR, "test", channels="rgb")

rgb_model = build_rgb_baseline().to(DEVICE)
rgb_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "rgb_baseline_best.pt")))
rgb_results = compute_iou_per_class(rgb_model, test_ds_rgb)
print_results("RGB baseline", rgb_results)

ms_model = build_multispectral_model().to(DEVICE)
ms_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "multispectral_best.pt")))
ms_results = compute_iou_per_class(ms_model, test_ds_ms)
print_results("Multispectral", ms_results)

print(f"\nDelta (multispectral - RGB) mean IoU: {ms_results['mean_iou'] - rgb_results['mean_iou']:.4f}")


=== RGB baseline ===
Pixel accuracy: 0.7682
Mean IoU (present classes): 0.3235
Per-class IoU:
  Tree cover          : 0.6464
  Shrubland           : 0.0000
  Grassland           : 0.1210
  Cropland            : 0.5216
  Built-up            : 0.7705
  Bare/sparse veg     : 0.0000
  Snow/ice            : 0.0000
  Water               : 0.5286
  Wetland             : 0.0000
  Mangroves           : 0.0000
  Moss/lichen         : 0.0000

=== Multispectral ===
Pixel accuracy: 0.7678
Mean IoU (present classes): 0.3313
Per-class IoU:
  Tree cover          : 0.6442
  Shrubland           : 0.0000
  Grassland           : 0.1334
  Cropland            : 0.4761
  Built-up            : 0.7785
  Bare/sparse veg     : 0.0000
  Snow/ice            : 0.0000
  Water               : 0.6185
  Wetland             : 0.0000
  Mangroves           : 0.0000
  Moss/lichen         : 0.0000

Delta (multispectral - RGB) mean IoU: 0.0078
